# BiRefNet and the SCHP parser on the GPU

## 1 · Environment

In [ ]:
import glob
import json
import os
import subprocess
import sys

DRIVE_PROJECT_DIR = "Side projects and shi"
PIP = [sys.executable, "-m", "pip"]
PROBE = "/content/ort_report.py"

ORT_REPORT = """
import ctypes, glob, json, os, site, sys
try:
    import onnxruntime as ort
except ImportError:
    print(json.dumps({"installed": False, "available": [], "loads": False, "error": "not installed"}))
    sys.exit(1)
info = {"installed": True, "version": ort.__version__, "device": ort.get_device(),
        "available": ort.get_available_providers(), "loads": False, "error": ""}
for d in sorted({d for p in site.getsitepackages() for d in glob.glob(os.path.join(p, "nvidia", "*", "lib"))}):
    for f in os.listdir(d):
        if ".so" in f and any(k in f for k in ("cudart", "cublas", "cudnn", "cufft", "curand")):
            try:
                ctypes.CDLL(os.path.join(d, f), mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass
so = glob.glob(os.path.dirname(ort.__file__) + "/capi/libonnxruntime_providers_cuda.so")
if not so:
    info["error"] = "wheel carries no CUDA provider library"
else:
    try:
        ctypes.CDLL(so[0], mode=ctypes.RTLD_GLOBAL)
        info["loads"] = True
    except OSError as e:
        info["error"] = str(e).strip().split("\\n")[-1]
print(json.dumps(info))
sys.exit(0 if info["loads"] else 1)
"""
open(PROBE, "w").write(ORT_REPORT)


def ort_report():
    r = subprocess.run([sys.executable, PROBE], capture_output=True, text=True)
    out = r.stdout.strip().splitlines()
    if out and out[-1].startswith("{"):
        return json.loads(out[-1])
    tail = (r.stderr.strip().splitlines() or ["probe produced no output"])[-1]
    return {"installed": True, "available": [], "loads": False, "error": tail[:120]}


import torch

before = ort_report()
print(f"{'gpu':28s} {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"{'torch CUDA':28s} {torch.version.cuda}")
print(f"{'onnxruntime':28s} {before.get('version', 'not installed')}")
print(f"{'ort device':28s} {before.get('device', '-')}")
print(f"{'registers CUDA provider':28s} {'CUDAExecutionProvider' in before['available']}")
print(f"{'CUDA provider actually loads':28s} {before['loads']}  {before['error']}")

## 2 · Install

In [ ]:
subprocess.run(PIP + ["install", "-q", "mediapipe==1.0.1", "huggingface_hub"], check=True, capture_output=True)
subprocess.run(PIP + ["uninstall", "-q", "-y", "onnxruntime", "onnxruntime-gpu"], capture_output=True)

chosen = None
for spec in ["==1.22.0", "", "==1.21.1", "==1.20.1", "==1.19.2", "==1.18.1"]:
    label = spec.lstrip("=") or "newest"
    if subprocess.run(PIP + ["install", "-q", f"onnxruntime-gpu{spec}"], capture_output=True).returncode:
        print(f"onnxruntime-gpu {label:8s} no installable wheel")
        continue
    info = ort_report()
    if info["loads"]:
        chosen = info["version"]
        print(f"onnxruntime-gpu {info['version']:8s} CUDA provider loads")
        break
    print(f"onnxruntime-gpu {info.get('version', label):8s} registers "
          f"{'CUDAExecutionProvider' in info['available']}, loads False - {info['error']}")
if chosen is None:
    raise RuntimeError("no onnxruntime-gpu build loads CUDA on this runtime")

subprocess.run(PIP + ["uninstall", "-q", "-y", "opencv-python", "opencv-python-headless",
                      "opencv-contrib-python"], capture_output=True)
subprocess.run(PIP + ["install", "-q", "opencv-contrib-python-headless==5.0.0.93"],
               check=True, capture_output=True)

import cv2

print(f"{'opencv':16s} {cv2.__version__}")
print(f"{'guidedFilter':16s} {hasattr(cv2.ximgproc, 'guidedFilter')}")

## 3 · Downloads

In [ ]:
import shutil
import zipfile

from google.colab import drive

BUNDLE = "https://github.com/101011101/magichour_takehome/raw/v3.3-lock/v33_ironman_bundle.zip"
shutil.rmtree("/content/gpu_check", ignore_errors=True)
subprocess.run(["wget", "-q", "-O", "/content/bundle.zip", BUNDLE], check=True)
with zipfile.ZipFile("/content/bundle.zip") as z:
    z.extractall("/content/gpu_check")
os.chdir("/content/gpu_check")
sys.path.insert(0, "lib")
for f in ("lib/garment_crop.py", "lib/phase3_variants.py", "lib/ironman_bc_crop.py"):
    if not os.path.exists(f):
        raise FileNotFoundError(f"bundle incomplete: {f}")

drive.mount("/content/drive")
MYDRIVE = "/content/drive/MyDrive"
BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
if not os.path.isdir(BASE):
    raise FileNotFoundError(f"Drive project dir not found: {BASE}")
cands = [os.path.join(MYDRIVE, "hf_cache"), os.path.join(BASE, "tryon_models", "hf_cache"),
         os.path.join(BASE, "hf_cache")]
os.environ["HF_HOME"] = next((c for c in cands if os.path.isdir(c)), cands[0])
os.environ["V3_MODEL_DIR"] = os.path.join(BASE, "v3_models")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["V3_MODEL_DIR"], exist_ok=True)
os.makedirs("/content/v2/runs/.models", exist_ok=True)
for name in ("BiRefNet_lite.onnx", "pose_landmarker_lite.task", "selfie_multiclass_256x256.tflite"):
    src = os.path.join(os.environ["V3_MODEL_DIR"], name)
    dst = f"/content/v2/runs/.models/{name}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)


def pick(pattern, exclude=None):
    zs = [z for z in sorted(glob.glob(os.path.join(BASE, "v3_runs", pattern)))
          if not (exclude and exclude in os.path.basename(z))]
    if not zs:
        raise FileNotFoundError(f"no {pattern} under Drive v3_runs/")
    return zs[-1]


for zp, suffix in ((pick("v34_ironman2_*.zip", exclude="_bc_"), "__bald.jpg"),
                   (pick("v34_ironman2_bc_*.zip"), "__BC.jpg")):
    with zipfile.ZipFile(zp) as z:
        members = [n for n in z.namelist() if n.endswith(suffix)]
        z.extractall("run", members=members)
    print(f"{os.path.basename(zp):44s} {len(members):3d} x {suffix}")
print(f"{'HF_HOME':44s} {os.environ['HF_HOME']}")

## 4 · Load and assert

In [ ]:
import numpy as np

import garment_crop as GC
import ironman_bc_crop as C
import phase3_variants as P


def sessions(device):
    os.environ["V2_ORT_GPU"] = "1" if device == "gpu" else "0"
    GC._STATE.pop("biref", None)
    GC._STATE.pop("biref_prov", None)
    P._HP.clear()
    GC._biref()
    P._parser()
    if P._HP.get("m") is None:
        raise RuntimeError("the SCHP parser did not load at all")
    return GC._STATE["biref_prov"], P._HP["m"].get_providers()[0]


biref_prov, parser_prov = sessions("gpu")
print(f"{'BiRefNet':16s} {biref_prov}")
print(f"{'SCHP parser':16s} {parser_prov}")
for name, prov in (("BiRefNet", biref_prov), ("SCHP parser", parser_prov)):
    if prov != "CUDAExecutionProvider":
        raise RuntimeError(f"{name} fell back to {prov} - the crops are not on the GPU")

## 5 · Timings

In [ ]:
import time

import cv2

probe = sorted(glob.glob("run/refs/*__bald.jpg"))[0]
img = cv2.imread(probe)


def timings(device):
    sessions(device)
    GC.biref_matte(img, f"warmup_{device}", True)
    P.parse_human(img)
    _, dt_biref = GC.biref_matte(img, f"timing_{device}", True)
    t = time.time()
    P.parse_human(img)
    return dt_biref, time.time() - t


gpu_biref, gpu_parser = timings("gpu")
cpu_biref, cpu_parser = timings("cpu")
print(f"{os.path.basename(probe)}  {img.shape[1]}x{img.shape[0]}\n")
print(f"{'stage':16s} {'GPU s':>8s} {'CPU s':>8s} {'speed-up':>9s}")
for name, g, c in (("BiRefNet", gpu_biref, cpu_biref), ("SCHP parser", gpu_parser, cpu_parser)):
    print(f"{name:16s} {g:8.2f} {c:8.2f} {c / g:8.1f}x")

## 6 · Parity against the CPU references of record — BUILD §7.3 acceptance test T1

In [ ]:
sessions("gpu")
shutil.rmtree(GC.CACHE_DIR, ignore_errors=True)

rows = []
for ref in sorted(glob.glob("run/refs/*__BC.jpg")):
    stem = os.path.basename(ref)[:-len("__BC.jpg")]
    bald = f"run/refs/{stem}__bald.jpg"
    if not os.path.exists(bald):
        continue
    a = cv2.imread(ref)
    b, cranium = C.crop_bc(cv2.imread(bald), f"gpu_{stem}")
    if abs(a.shape[0] - b.shape[0]) > 8 or abs(a.shape[1] - b.shape[1]) > 8:
        rows.append((stem, float("nan"), float("nan"), False, cranium,
                     f"shape {b.shape[1]}x{b.shape[0]} vs {a.shape[1]}x{a.shape[0]}"))
        continue
    d = np.abs(a.astype(np.float32) - cv2.resize(b, (a.shape[1], a.shape[0])).astype(np.float32))
    rows.append((stem, float(d.mean()), float(d.max()), float(d.mean()) <= 4.0, cranium, ""))

failed = [r for r in rows if not r[3]]
print(f"{'reference':54s} {'MAD':>7s} {'max':>5s}  cranium  note")
for stem, mad, mx, ok, cranium, note in rows:
    print(f"{stem[:54]:54s} {mad:7.2f} {mx:5.0f}  {str(cranium):7s}  {note}")
print(f"\n{len(rows)} references · {len(rows) - len(failed)} within MAD 4.0 · {len(failed)} outside")

## 7 · Verdict

In [ ]:
mads = [r[1] for r in rows if r[3]]
parity = bool(rows) and not failed
print(f"{'onnxruntime-gpu':30s} {chosen}")
print(f"{'BiRefNet provider':30s} {biref_prov}")
print(f"{'SCHP parser provider':30s} {parser_prov}")
print(f"{'BiRefNet GPU / CPU':30s} {gpu_biref:.2f}s / {cpu_biref:.2f}s  {cpu_biref / gpu_biref:.1f}x")
print(f"{'SCHP parser GPU / CPU':30s} {gpu_parser:.2f}s / {cpu_parser:.2f}s  {cpu_parser / gpu_parser:.1f}x")
print(f"{'crop parity (T1)':30s} {'PASS' if parity else 'FAIL'} - "
      f"{len(rows) - len(failed)}/{len(rows)} within MAD 4.0, worst {max(mads) if mads else float('nan'):.2f}")
print(f"{'run crops on GPU in production':30s} {'yes' if parity else 'no - keep crops on CPU'}")